In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

data = pd.read_csv("../datasets/Indoor_Plant_Health_and_Growth_Factors.csv")
data.sample(5)

feature_columns = [
    'Plant_ID',
    'Height_cm',
    'Leaf_Count',
    'New_Growth_Count',
    'Watering_Amount_ml',
    'Watering_Frequency_days',
    'Light_Intensity',
    'Ambient_Temperature',
    'Humidity',
    'Fertilizer_Type',
    'Fertilizer_Amount_ml',
    'Pest_Presence',
    'Pest_Severity',
    'Soil_Moisture',
    'Soil_Type'
    ]

target_column = 'Plant_Health_Status'

required_columns = feature_columns + [target_column]

missing_columns = [col for col in required_columns if col not in data.columns]

if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

print("Dataset shape:", data.shape)
print("\nColumn names:")
print(data.columns.tolist())

print("\nData types:")
print(data[required_columns].dtypes)

print("\nMissing values per required column:")
print(data[required_columns].isnull().sum())

print("\nDuplicate rows:", data.duplicated().sum())

In [ ]:
dataset = data[required_columns].copy()
dataset[target_column] = dataset[target_column]

dataset = dataset.dropna(subset=[target_column])

print("\nNew target distribution:")
print(dataset[target_column].value_counts())

dataset.sample(5)

In [ ]:
print("\nTarget class distribution:")
print(dataset[target_column].value_counts())

dataset[target_column].value_counts().plot(kind='bar', figsize=(7, 4))
plt.title("Plant Health Class Distribution")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
x = dataset[feature_columns]
x['Pest_Presence'] = x['Pest_Presence'].fillna("None")
x['Pest_Severity'] = x['Pest_Presence'].fillna("None")
x = pd.get_dummies(x, columns=['Plant_ID'])
x = pd.get_dummies(x, columns=['Light_Intensity'])
x = pd.get_dummies(x, columns=['Fertilizer_Type'])
x = pd.get_dummies(x, columns=['Pest_Presence'])
x = pd.get_dummies(x, columns=['Pest_Severity'])
x = pd.get_dummies(x, columns=['Soil_Type'])
y = dataset[target_column]

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(random_state=42)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
print("Train set report:")
train_predictions = model.predict(x_train)
print(classification_report(y_train, train_predictions))

print("Test set report:")
test_predictions = model.predict(x_test)
print(classification_report(y_test, test_predictions))

In [ ]:
print("Dataset Validation Summary")
print("--------------------------")
print(f"Rows used: {len(dataset)}")
print(f"Features used: {feature_columns}")
print(f"Target classes: {sorted(y.unique().tolist())}")
print(f"Duplicate rows in original data: {data.duplicated().sum()}")
print(f"Test accuracy: {accuracy_score(y_test, test_predictions):.4f}")